# Tutorial 1 - First steps with ParlaMint dataset

Welcome to this **hands-on tutorial exploring the ParlaMint dataset**, a rich multilingual corpus of European parliamentary debates. This tutorial is designed for researchers, data scientists and political science enthusiasts who want to analyze parliamentary proceedings across different European countries. 

The **ParlaMint dataset** is a comprehensive and multilingual corpus of transcribed speeches from different European parliaments. It serves as a valuable resource for computational social science and digital humanities research. 

At its core, ParlaMint contains the full text of parliamentary speeches annotated with rich metadata, including the speaker's role (e.g. *Member of Parliament*), political party affiliation, date of the session and the length of each speech. This foundational data was significantly expanded with **CAP categories** and **sentiment scores**.

First, the ParlaCAP component categorizes all available speech segments into a specific policy domain from the **Comparative Agendas Project (CAP)**, e.g. healthcare, education or foreign affairs. 

Second, the **ParlaSent** extension provides detailed **sentiment scores** for each segment, allowing researchers to analyze the emotional tone of the debates.

The dataset's power lies in its comparative design. It covers **multiple countries** and spans different time periods, enabling cross-parliament analysis. This makes ParlaMint very useful for **comparative political research**, allowing users to systematically study the differences in e.g. political discourse or policy priorities, across nations. Besides that, its structure makes it ideal for technical applications like **topic modeling** and **sentiment analysis** on a large scale.

*Source: Erjavec, Tomaž; et al. (2025). Multilingual comparable corpora of parliamentary debates ParlaMint 5.0. Slovenian language resource repository CLARIN.SI. ISSN 2820-4042. http://hdl.handle.net/11356/2004*


## Before you start

Before running this notebook, make sure Tutorial 0 works and returns one API row. The goal here is not to analyze everything at once, but to get a safe first look at the data.

## What this notebook produces

By the end of this notebook, you will have:
- a small dataframe called `filtered_all`
- a first look at the most important columns
- examples of `head()`, `shape()`, `describe()`, `unique()`, and `value_counts()`

## What to do if a cell fails

- rerun the import cell first
- check that `my_secrets.env` exists in the repo root
- if the API request fails, go back to Tutorial 0 and confirm the key still works

You do not need to understand every line of code on the first read. Focus on the **input**, the **output**, and what each step changes in the dataframe.


**Importing Libraries**

We only need a few tools here.


In [1]:
import os
import pandas as pd
import requests

**Load a small sample**

We will load a small sample from the API, so the notebook stays fast and simple.


This notebook keeps the example intentionally small. We fetch the data from the API in pages so the request stays manageable and the notebook remains beginner-friendly.

In plain language:
- `limit=1000` asks for up to 1000 rows at a time
- `offset` tells the API where the next page should start
- the loop stops when the API returns no more rows, or when the notebook reaches its safety cutoff

After the loading cell runs, use `filtered_all.head()` and `filtered_all.shape` as quick sanity checks:
- `head()` shows what the first rows look like
- `shape` tells you how many rows and columns you loaded


In [2]:
url = "https://parlacap.ipipan.waw.pl/api/v1/"

os.environ["PARLACAP_API_KEY"] = open("../my_secrets.env", encoding="utf-8").read().split("=", 1)[1].strip()
headers = {"X-API-Key": os.getenv("PARLACAP_API_KEY")}

# The API returns data in pages.
# `limit` is the page size and `offset` moves us to the next page.
# This keeps the example lightweight for a tutorial notebook.

# We load the data in small steps.
i = 0
data = []

while True:
    # Stop when there are no more rows, or when the safety cutoff is reached.
    if i > 10_000:
        break

    try:
        filter_data = {
            "limit": 1000,
            "offset": i,
            "columns": ["cap_category", "date", "sent_logit", "parliament"],
        }

        response = requests.post(url + "filter", json=filter_data, headers=headers)
        if response.status_code != 200:
            raise Exception(f"Got weird response code: {response.content}")

        payload = response.json()["rows"]
        if not payload:
            break

        data.extend(payload)
        i = i + 1000
        print(f"{i=}")

        if len(payload) < 1000:
            break
    except Exception as e:
        raise (e)

# Turn the rows into a table.
filtered_all = pd.DataFrame.from_records(data).rename(columns={"cap_category": "CAP_category", "parliament": "country"})

if filtered_all.empty:
    raise ValueError("No data came back. Check the API key.")

# Clean the table a little bit.
filtered_all["date"] = pd.to_datetime(filtered_all["date"], errors="coerce")

filtered_all = filtered_all.dropna(subset=["CAP_category", "date", "sent_logit"])
filtered_all = filtered_all[~filtered_all["CAP_category"].isin(["Mix", "Other"])].copy()

filtered_all.shape


i=1000
i=2000
i=3000
i=4000
i=5000
i=6000
i=7000
i=8000
i=9000
i=10000
i=11000


(4874, 4)

In [3]:
filtered_all.head(10)

,CAP_category,date,sent_logit,country
7,Government Operations,1996-01-15,2.973,AT
9,Government Operations,1996-01-15,2.356,AT
13,Government Operations,1996-01-15,1.798,AT
15,Civil Rights,1996-01-15,1.703,AT
17,Government Operations,1996-01-15,2.034,AT
29,Government Operations,1996-01-15,2.899,AT
44,Government Operations,1996-01-15,2.855,AT
45,Government Operations,1996-01-15,2.705,AT
46,Government Operations,1996-01-15,2.925,AT
47,Government Operations,1996-01-15,2.926,AT


**3.2. shape()**

Our next question is: **How much data are we working with?**

The '.shape()' method answers this by outputting a pair of numbers in this format: '(number_of_rows, number_of_columns)'.


The output, e.g. (2754914, 21), would tell us that we are analyzing **2.754.914 rows (individual speeches)**, each described by **21 different variables**. 


In [4]:
filtered_all[["country", "CAP_category", "sent_logit"]].describe(include="all")

,country,CAP_category,sent_logit
count,4874,4874,4874.000000
unique,1,21,NaN
top,AT,Macroeconomics,NaN
freq,4874,637,NaN
mean,NaN,NaN,2.118163
std,NaN,NaN,0.846999
min,NaN,NaN,-0.033000
25%,NaN,NaN,1.446250
50%,NaN,NaN,2.126000
75%,NaN,NaN,2.758750


**3.4. unique()**

The 'unique()' method extracts a clean list of all possible values in a colum (e.g. 'CAP_category'). It extracts every distinct category without any duplicates or counts.

When we wrap that method in 'pd.Series()', it outputs a list in an easy-to-read format.


Here, we do the same but instead of 'CAP_category', we look at all unique values in the column 'party_status'.


... or 'party_orientation'.


**3.5. value_counts()**

To understand the data on a deeper level, we use '.value_counts()'. This method **counts how many times each unique value appears** in a column. Also, it shows us what values are the most common.


In the case of 'CAP_category', the command above answers the question: **"Which policy topics dominate the parliamentary agenda?"**. The output is a ranked list, showing the most frequently debated topics at the top. 

*The same method can be applied to other columns, such as 'party_orientation' to see the left-right balance of speeches.*


In [5]:
(
    filtered_all["CAP_category"].isnull().any(),
    filtered_all["date"].isnull().any(),
    filtered_all["sent_logit"].isnull().any(),
)

(np.False_, np.False_, np.False_)

**Conclusion**

In this first tutorial, we took the essential steps toward working with the ParlaMint dataset. We learned how to:
- **Set up the environment** by installing and importing the necessary Python libraries
- **Load and filter the data efficiently**, using memory optimization techniques and chunked processing to handle very large files.
- **Clean and refine the dataset**
- **Explore the data** with foundational 'pandas'-methods such as '.head()', '.shape()', '.describe()', 'unique()' and '.value_counts()' to get a first overview of the structure and content

By the end of this tutorial, you now have a **clean, filtered dataset** of parliamentary debates that is ready for deeper analysis. 

In the next tutorial (**Tutorial 2**), we will take this filtered dataset and explore **topic and sentiment distributions** in depth - visualizing which CAP categories dominate parliamentary debates or how sentiment varies by topic and country.
